## Dataset Extraction

## Imports

In [2]:
import pandas as pd

In [20]:
tracks = pd.read_csv('data/tracks.csv')
tracks.columns

/tmp/ipykernel_58377/1475502056.py:1: DtypeWarning: Columns (0,1,5,6,8,12,18,20,21,22,24,33,34,38,39,44,47,49) have mixed types. Specify dtype option on import or set low_memory=False.
  tracks = pd.read_csv('data/tracks.csv')


Index(['Unnamed: 0', 'album', 'album.1', 'album.2', 'album.3', 'album.4',
       'album.5', 'album.6', 'album.7', 'album.8', 'album.9', 'album.10',
       'album.11', 'album.12', 'artist', 'artist.1', 'artist.2', 'artist.3',
       'artist.4', 'artist.5', 'artist.6', 'artist.7', 'artist.8', 'artist.9',
       'artist.10', 'artist.11', 'artist.12', 'artist.13', 'artist.14',
       'artist.15', 'artist.16', 'set', 'set.1', 'track', 'track.1', 'track.2',
       'track.3', 'track.4', 'track.5', 'track.6', 'track.7', 'track.8',
       'track.9', 'track.10', 'track.11', 'track.12', 'track.13', 'track.14',
       'track.15', 'track.16', 'track.17', 'track.18', 'track.19'],
      dtype='object')

## Genre Analysis

In [21]:
#we'll use 8 general classes for the genre
#country & blues, classical & jazz are outnumbered, and we want to avoid class imbalance->remove them from the dataset
genres = ["Hip-Hop", "Electronic", "Instrumental", "Pop", "Folk", "Rock", "Experimental", "International"]
genres_df= pd.read_csv('data/genres.csv')

main_genres_df = genres_df[genres_df["title"].str.contains('|'.join(genres), case=False, na=False)][["title", "genre_id", "parent"]]
main_genres_df.head()

,title,genre_id,parent
1,International,2,0
9,Pop,10,0
11,Rock,12,0
14,Electronic,15,0
16,Folk,17,0


In [22]:
parent_genres_df = main_genres_df[main_genres_df["parent"] == 0].head(20)
genre_values = parent_genres_df.head(10).genre_id.values
genre_values

array([   2,   10,   12,   15,   17,   21,   38, 1235])

In [23]:
track_columns = [
    'track_id','comments', 'date_created', 'date_released', 'engineer', 'favorites', 
    'id', 'information', 'listens', 'producer', 'tags', 'title', 'tracks', 
    'type', 'active_year_begin', 'active_year_end', 'associated_labels', 
    'bio', 'comments', 'date_created', 'favorites', 'id', 'latitude', 
    'location', 'longitude', 'members', 'name', 'related_projects', 
    'tags', 'website', 'wikipedia_page', 'split', 'subset', 'bit_rate', 
    'comments', 'composer', 'date_created', 'date_recorded', 'duration', 
    'favorites', 'genre_top', 'genres', 'genres_all', 'information', 
    'interest', 'language_code', 'license', 'listens', 'lyricist', 
    'number', 'publisher', 'tags', 'title'
]
track_column_map = dict(zip([col for col in tracks.columns], track_columns))

tracks.rename(columns=track_column_map, inplace=True)
tracks = tracks[tracks["subset"].isin(["small"])]

In [24]:
tracks.genre_top.value_counts()

genre_top
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

## Balanced Sampling From FMA Dataset

In [25]:
def map_genre_to_id(genre_str):
    if pd.isna(genre_str):
        return None
    return main_genres_df[main_genres_df["title"] == genre_str]["genre_id"].values[0]

In [26]:
filtered_tracks = tracks[tracks["genre_top"].isin(main_genres_df["title"])]
filtered_tracks["genre_top_id"] = filtered_tracks["genre_top"].apply(map_genre_to_id)

clf_tracks_df = filtered_tracks[filtered_tracks["genre_top_id"].isin(genre_values)][["track_id", "genre_top_id","genre_top"]]

In [27]:
clf_tracks_df["genre_top"].value_counts()    

genre_top
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

## Sample 250 Instances per Genre

In [28]:
idx_arr = []
sample_size = 250
for genre_id in genre_values: 
    sample_genre_idx = clf_tracks_df[clf_tracks_df["genre_top_id"] == genre_id].sample(sample_size).index
    idx_arr.extend(sample_genre_idx)

balanced_clf_tracks_df = clf_tracks_df[clf_tracks_df.index.isin(idx_arr)]
balanced_clf_tracks_df["genre_top"].value_counts()

genre_top
Folk             250
Rock             250
Hip-Hop          250
International    250
Pop              250
Experimental     250
Electronic       250
Instrumental     250
Name: count, dtype: int64

In [29]:
balanced_clf_tracks_df.to_csv("data/fma_tracks.csv", index=False)

## Training Data Creation For ML & DNN Models

## Dataset Generation for ML Models

In [5]:
import librosa as lr
import numpy as np
import os

## Create DF For GTZAN Tracks

In [31]:
mfcc_genres = os.listdir("data/gtzan/genres")
genre_mapping = {genre: i for i, genre in enumerate(mfcc_genres)}
mfcc_data_df = pd.DataFrame(columns=["track_id", "genre_id", "genre_name"])

for genre in mfcc_genres:
    genre_dir = os.path.join("data/gtzan/genres", genre)
    for file_name in os.listdir(genre_dir):
        if file_name.endswith(".au"):
            genre, track_id, _ = file_name.split(".")
            genre_id = genre_mapping[genre]
            track_row = pd.DataFrame({
                "track_id": [track_id],
                "genre_id": [genre_id],
                "genre_name": [genre]
            })
            mfcc_data_df = pd.concat([mfcc_data_df, track_row], ignore_index=True)

In [32]:
mfcc_data_df.genre_name.value_counts()

genre_name
classical    100
metal        100
disco        100
pop          100
reggae       100
jazz         100
hiphop       100
blues        100
country      100
rock         100
Name: count, dtype: int64

In [33]:
mfcc_data_df.to_csv("data/gtzan_tracks.csv", index=False)

## Create 1 Sample / Excerpt Feature DF for GTZAN & FMA Datasets

For each 30-second audio excerpt, we'll do:
- divide it into 10 3-second excerpts
- For each 3-second excerpt
    - compute features + take mean & variance
    - add the feature row to data frame

## Define Utility Functions

In [87]:
#Derive pd dataframe columns 
def flatten_features(d):
    flat_dict = {}
    for k, v in d.items():
        if isinstance(v, (np.ndarray, list)) and len(np.atleast_1d(v)) > 1:
            # Create columns like chroma_cqt_mean_0, chroma_cqt_mean_1...
            for i, val in enumerate(v):
                flat_dict[f"{k}_{i}"] = [val]
        else:
            # Handle scalars or single-element arrays
            flat_dict[k] = [float(v) if isinstance(v, np.ndarray) else v]
    return flat_dict

In [ ]:
# Example for a sample 30 second audio
FMA_FOLDER_PATH = "data/fma_small/"
GTZAN_FOLDER_PATH = "data/gtzan/genres/"
FS = 22050
TARGET_DURATION = 30
TOTAL_SAMPLES = FS * TARGET_DURATION

def fma_row_to_file_path(row):
    track_id = row["track_id"]
    return os.path.join(FMA_FOLDER_PATH, f"{track_id//1000:03d}", f"{track_id:06d}.mp3")

def gtzan_row_to_file_path(row):
    genre = row["genre_name"]
    track_id = row["track_id"]
    return os.path.join(GTZAN_FOLDER_PATH, genre, f"{genre}.{track_id:05d}.au")

def extract_ml_features(row, nfft=1024, hop_length=512, n_mfcc=13, isGTZAN=True):
    if isGTZAN:
        file_path = gtzan_row_to_file_path(row)
        try:
            y, sr = lr.load(file_path, duration=TARGET_DURATION, sr=FS, dtype=float)
        except Exception as e:
            print(f"Skipping broken file {file_path}: {e}")
            return None
    else:
        file_path = fma_row_to_file_path(row)
        try:
            y, sr = lr.load(file_path, duration=TARGET_DURATION, sr=FS, dtype=float)
        except Exception as e:
            print(f"Skipping broken file {file_path}: {e}")
            return None
        
    if len(y) < TOTAL_SAMPLES:
            # Calculate how many zeros we need to reach 30s
            padding = TOTAL_SAMPLES - len(y)
            y = np.pad(y, (0, padding), mode='constant')
            
    # compute all necessary features here, spectral, tempo, rhythm etc.
    # extract: chrome_cqt, chroma_cens, tonnetz, rms, spectral_centroid, spectral_bandwidth, spectral_rolloff, zero_crossing_rate, tempo, mfccs
    features = {}
    features["tempo"] = lr.feature.tempo(y=y, sr=sr,hop_length=hop_length)[0]
    chroma_cqt = lr.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)
    features["chroma_cqt_mean"] = chroma_cqt.mean(axis=1)
    features["chroma_cqt_var"] = chroma_cqt.var(axis=1)

    chroma_cens = lr.feature.chroma_cens(y=y, sr=sr, hop_length=hop_length)
    features["chroma_cens_mean"] = chroma_cens.mean(axis=1)
    features["chroma_cens_var"] = chroma_cens.var(axis=1)

    features["tonnetz"] = lr.feature.tonnetz(y=lr.effects.harmonic(y, n_fft=nfft, hop_length=hop_length), sr=sr).mean(axis=1)
    
    rms = lr.feature.rms(y=y, frame_length=nfft, hop_length=hop_length)
    features["rms_mean"] = rms.mean(axis=1)
    features["rms_var"] = rms.var(axis=1)

    spectral_centroid = lr.feature.spectral_centroid(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_centroid_mean"] = spectral_centroid.mean(axis=1)
    features["spectral_centroid_var"] = spectral_centroid.var(axis=1)

    spectral_bandwidth = lr.feature.spectral_bandwidth(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_bandwidth_mean"] = spectral_bandwidth.mean(axis=1)
    features["spectral_bandwidth_var"] = spectral_bandwidth.var(axis=1)

    spectral_rolloff = lr.feature.spectral_rolloff(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_rolloff_mean"] = spectral_rolloff.mean(axis=1)
    features["spectral_rolloff_var"] = spectral_rolloff.var(axis=1)

    zero_crossing_rate = lr.feature.zero_crossing_rate(y, frame_length=nfft)
    features["zero_crossing_rate_mean"] = zero_crossing_rate.mean(axis=1)
    features["zero_crossing_rate_var"] = zero_crossing_rate.var(axis=1)

    mfccs = lr.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=nfft, hop_length=hop_length)

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = mfccs[i].mean()
        features[f"mfcc_{i+1}_var"] = mfccs[i].var()

    if isGTZAN:
        row_dict = {
            "track_id": row["track_id"],"genre_id": row["genre_id"],"genre_name": row["genre_name"]
        }
    else:
        row_dict = {
            "track_id": row["track_id"],"genre_top_id": row["genre_top_id"],"genre_top": row["genre_top"]
        }
    feature_dict = {key: value for key, value in features.items()}
    row_dict = row_dict | feature_dict
    return row_dict

In [89]:
gtzan_tracks = pd.read_csv("data/gtzan_tracks.csv")
gtzan_columns = {}
for index, row in gtzan_tracks.iterrows():
    features_dict = extract_ml_features(row)
    flat_dict = flatten_features(features_dict)
    gtzan_columns = flat_dict.keys()
    print(gtzan_columns)
    break

dict_keys(['track_id', 'genre_id', 'genre_name', 'tempo', 'chroma_cqt_mean_0', 'chroma_cqt_mean_1', 'chroma_cqt_mean_2', 'chroma_cqt_mean_3', 'chroma_cqt_mean_4', 'chroma_cqt_mean_5', 'chroma_cqt_mean_6', 'chroma_cqt_mean_7', 'chroma_cqt_mean_8', 'chroma_cqt_mean_9', 'chroma_cqt_mean_10', 'chroma_cqt_mean_11', 'chroma_cqt_var_0', 'chroma_cqt_var_1', 'chroma_cqt_var_2', 'chroma_cqt_var_3', 'chroma_cqt_var_4', 'chroma_cqt_var_5', 'chroma_cqt_var_6', 'chroma_cqt_var_7', 'chroma_cqt_var_8', 'chroma_cqt_var_9', 'chroma_cqt_var_10', 'chroma_cqt_var_11', 'chroma_cens_mean_0', 'chroma_cens_mean_1', 'chroma_cens_mean_2', 'chroma_cens_mean_3', 'chroma_cens_mean_4', 'chroma_cens_mean_5', 'chroma_cens_mean_6', 'chroma_cens_mean_7', 'chroma_cens_mean_8', 'chroma_cens_mean_9', 'chroma_cens_mean_10', 'chroma_cens_mean_11', 'chroma_cens_var_0', 'chroma_cens_var_1', 'chroma_cens_var_2', 'chroma_cens_var_3', 'chroma_cens_var_4', 'chroma_cens_var_5', 'chroma_cens_var_6', 'chroma_cens_var_7', 'chroma_cens

/tmp/ipykernel_59313/621378076.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  flat_dict[k] = [float(v) if isinstance(v, np.ndarray) else v]


In [90]:
fma_tracks = pd.read_csv("data/fma_tracks.csv")
fma_columns = {}
for index, row in fma_tracks.iterrows():
    features_dict = extract_ml_features(row, isGTZAN=False)
    flat_dict = flatten_features(features_dict)
    fma_columns = flat_dict.keys()
    print(fma_columns)
    break

dict_keys(['track_id', 'genre_top_id', 'genre_top', 'tempo', 'chroma_cqt_mean_0', 'chroma_cqt_mean_1', 'chroma_cqt_mean_2', 'chroma_cqt_mean_3', 'chroma_cqt_mean_4', 'chroma_cqt_mean_5', 'chroma_cqt_mean_6', 'chroma_cqt_mean_7', 'chroma_cqt_mean_8', 'chroma_cqt_mean_9', 'chroma_cqt_mean_10', 'chroma_cqt_mean_11', 'chroma_cqt_var_0', 'chroma_cqt_var_1', 'chroma_cqt_var_2', 'chroma_cqt_var_3', 'chroma_cqt_var_4', 'chroma_cqt_var_5', 'chroma_cqt_var_6', 'chroma_cqt_var_7', 'chroma_cqt_var_8', 'chroma_cqt_var_9', 'chroma_cqt_var_10', 'chroma_cqt_var_11', 'chroma_cens_mean_0', 'chroma_cens_mean_1', 'chroma_cens_mean_2', 'chroma_cens_mean_3', 'chroma_cens_mean_4', 'chroma_cens_mean_5', 'chroma_cens_mean_6', 'chroma_cens_mean_7', 'chroma_cens_mean_8', 'chroma_cens_mean_9', 'chroma_cens_mean_10', 'chroma_cens_mean_11', 'chroma_cens_var_0', 'chroma_cens_var_1', 'chroma_cens_var_2', 'chroma_cens_var_3', 'chroma_cens_var_4', 'chroma_cens_var_5', 'chroma_cens_var_6', 'chroma_cens_var_7', 'chroma_c

/tmp/ipykernel_59313/621378076.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  flat_dict[k] = [float(v) if isinstance(v, np.ndarray) else v]


In [93]:
gtzan_features_df = pd.DataFrame(columns=list(gtzan_columns))
fma_features_df = pd.DataFrame(columns=list(fma_columns))

## Fill the Feature Arrays

In [ ]:
from datetime import datetime
start_time = datetime.now()

for index, row in gtzan_tracks.iterrows():
    feature_dict = extract_ml_features(row, isGTZAN=True)
    if feature_dict is None:
        print("Broken audio file, continuing")
        continue
    flat_feature_dict = flatten_features(feature_dict)
    feature_row = pd.DataFrame(flat_feature_dict)
    gtzan_features_df = pd.concat([gtzan_features_df, feature_row], ignore_index=True)
    if index % 20 == 0:
        end_time = datetime.now()
        difference = (end_time - start_time).total_seconds()
        print(f"#Samples Processed: {index} in {difference // 60} minutes {round(difference) % 60} seconds")

for index, row in fma_tracks.iterrows():
    feature_dict = extract_ml_features(row, isGTZAN=False)
    if feature_dict is None:
        print("Broken audio file, continuing")
        continue
    flat_feature_dict = flatten_features(feature_dict)
    feature_row = pd.DataFrame(flat_feature_dict)
    fma_features_df = pd.concat([fma_features_df, feature_row], ignore_index=True)
    if index % 20 == 0:
        end_time = datetime.now()
        difference = (end_time - start_time).total_seconds()
        print(f"#Samples Processed: {index} in {difference // 60} minutes {round(difference) % 60} seconds")
        
fma_features_df.to_csv("data/features/fma/fma_features.csv")
gtzan_features_df.to_csv("data/features/gtzan/gtzan_features.csv")

In [102]:
fma_features_df.shape

(2012, 94)

In [ ]:
gtzan_features_df.head()

,track_id,genre_id,genre_name,tempo,chroma_cqt_mean_0,chroma_cqt_mean_1,chroma_cqt_mean_2,chroma_cqt_mean_3,chroma_cqt_mean_4,chroma_cqt_mean_5,...,mfcc_9_mean,mfcc_9_var,mfcc_10_mean,mfcc_10_var,mfcc_11_mean,mfcc_11_var,mfcc_12_mean,mfcc_12_var,mfcc_13_mean,mfcc_13_var
0,72,0,classical,117.453835,0.345143,0.349426,0.453278,0.335365,0.247940,0.458865,...,3.862633,69.634758,1.341745,126.327907,1.302141,221.927448,5.317878,196.848944,3.427349,82.835012
1,72,0,classical,117.453835,0.345143,0.349426,0.453278,0.335365,0.247940,0.458865,...,3.862633,69.634758,1.341745,126.327907,1.302141,221.927448,5.317878,196.848944,3.427349,82.835012
2,72,0,classical,117.453835,0.345143,0.349426,0.453278,0.335365,0.247940,0.458865,...,3.862633,69.634758,1.341745,126.327907,1.302141,221.927448,5.317878,196.848944,3.427349,82.835012
3,73,0,classical,129.199219,0.526168,0.213332,0.380645,0.217537,0.239835,0.588029,...,-7.817801,92.806664,0.162575,79.366220,1.071013,163.209663,5.442096,230.326749,8.061357,200.185232
4,17,0,classical,117.453835,0.339787,0.377161,0.379565,0.250083,0.677330,0.308804,...,-9.648018,88.357251,1.319806,138.594191,-4.928373,165.889575,5.879800,155.297879,4.033052,202.983901


## Create a 10 sample/Excerpt Feature Dataset

For each 30-second audio excerpt, we'll do:
- divide it into 10 3-second excerpts
- For each 3-second excerpt
    - compute features + take mean & variance
    - add the feature row to data frame
- we'll compute in total of 10 inputs for each audio excerpt

In [ ]:
import librosa as lr
import numpy as np
import os
import pandas as pd

In [ ]:
def flatten_features(d):
    flat_dict = {}
    for k, v in d.items():
        if isinstance(v, (np.ndarray, list)) and len(np.atleast_1d(v)) > 1:
            # Create columns like chroma_cqt_mean_0, chroma_cqt_mean_1...
            for i, val in enumerate(v):
                flat_dict[f"{k}_{i}"] = [val]
        else:
            # Handle scalars or single-element arrays
            flat_dict[k] = [float(v) if isinstance(v, np.ndarray) else v]
    return flat_dict

FMA_FOLDER_PATH = "data/fma_small/"
GTZAN_FOLDER_PATH = "data/gtzan/genres/"
FS = 22050
TARGET_DURATION = 30
TOTAL_SAMPLES = FS * TARGET_DURATION

def fma_row_to_file_path(row):
    track_id = row["track_id"]
    return os.path.join(FMA_FOLDER_PATH, f"{track_id//1000:03d}", f"{track_id:06d}.mp3")

def gtzan_row_to_file_path(row):
    genre = row["genre_name"]
    track_id = row["track_id"]
    return os.path.join(GTZAN_FOLDER_PATH, genre, f"{genre}.{track_id:05d}.au")

In [ ]:
def extract_ml_features(row, nfft=1024, hop_length=512, n_mfcc=13, isGTZAN=True, chunk_number=1):
    if isGTZAN:
        file_path = gtzan_row_to_file_path(row)
        try:
            y, sr = lr.load(file_path, duration=TARGET_DURATION, sr=FS, dtype=float)
        except Exception as e:
            print(f"Skipping broken file {file_path}: {e}")
            return None
    else:
        file_path = fma_row_to_file_path(row)
        try:
            y, sr = lr.load(file_path, duration=TARGET_DURATION, sr=FS, dtype=float)
        except Exception as e:
            print(f"Skipping broken file {file_path}: {e}")
            return None
        
    if len(y) < TOTAL_SAMPLES:
            # Calculate how many zeros we need to reach 30s
            padding = TOTAL_SAMPLES - len(y)
            y = np.pad(y, (0, padding), mode='constant')

    chunk_length = int(TARGET_DURATION / 10) * sr
    y = y[(chunk_number - 1) * chunk_length : chunk_number * chunk_length]

    # compute all necessary features here, spectral, tempo, rhythm etc.
    # extract: chrome_cqt, chroma_cens, tonnetz, rms, spectral_centroid, spectral_bandwidth, spectral_rolloff, zero_crossing_rate, tempo, mfccs
    
    features = {}
    features["tempo"] = lr.feature.tempo(y=y, sr=sr,hop_length=hop_length)[0]
    chroma_cqt = lr.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)
    features["chroma_cqt_mean"] = chroma_cqt.mean(axis=1)
    features["chroma_cqt_var"] = chroma_cqt.var(axis=1)

    chroma_cens = lr.feature.chroma_cens(y=y, sr=sr, hop_length=hop_length)
    features["chroma_cens_mean"] = chroma_cens.mean(axis=1)
    features["chroma_cens_var"] = chroma_cens.var(axis=1)

    features["tonnetz"] = lr.feature.tonnetz(y=lr.effects.harmonic(y, n_fft=nfft, hop_length=hop_length), sr=sr).mean(axis=1)
    
    rms = lr.feature.rms(y=y, frame_length=nfft, hop_length=hop_length)
    features["rms_mean"] = rms.mean(axis=1).item()
    features["rms_var"] = rms.var(axis=1).item()

    spectral_centroid = lr.feature.spectral_centroid(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_centroid_mean"] = spectral_centroid.mean(axis=1).item()
    features["spectral_centroid_var"] = spectral_centroid.var(axis=1).item()

    spectral_bandwidth = lr.feature.spectral_bandwidth(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_bandwidth_mean"] = spectral_bandwidth.mean(axis=1).item()
    features["spectral_bandwidth_var"] = spectral_bandwidth.var(axis=1).item()

    spectral_rolloff = lr.feature.spectral_rolloff(y=y, sr=sr, n_fft=nfft, hop_length=hop_length)
    features["spectral_rolloff_mean"] = spectral_rolloff.mean(axis=1).item()
    features["spectral_rolloff_var"] = spectral_rolloff.var(axis=1).item()

    zero_crossing_rate = lr.feature.zero_crossing_rate(y, frame_length=nfft)
    features["zero_crossing_rate_mean"] = zero_crossing_rate.mean(axis=1).item()
    features["zero_crossing_rate_var"] = zero_crossing_rate.var(axis=1).item()

    mfccs = lr.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=nfft, hop_length=hop_length)

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = mfccs[i].mean()
        features[f"mfcc_{i+1}_var"] = mfccs[i].var()

    if isGTZAN:
        row_dict = {
            "track_id": int(f"{row['track_id']}{chunk_number - 1}"),"genre_id": row["genre_id"],"genre_name": row["genre_name"]
        }
    else:
        row_dict = {
            "track_id": int(f"{row['track_id']}{chunk_number - 1}"),"genre_top_id": row["genre_top_id"],"genre_top": row["genre_top"]
        }
    feature_dict = {key: value for key, value in features.items()}
    row_dict = row_dict | feature_dict
    return row_dict

## Define 3_sec_feature Data Frames

In [ ]:
gtzan_features_df = pd.read_csv("data/features/gtzan/gtzan_features.csv")
fma_features_df = pd.read_csv("data/features/fma/fma_features.csv")

gtzan_3sec_features_df = pd.DataFrame(columns=gtzan_features_df.columns)
fma_3sec_features_df = pd.DataFrame(columns=fma_features_df.columns)

gtzan_tracks = pd.read_csv("data/gtzan_tracks.csv")
fma_tracks = pd.read_csv("data/fma_tracks.csv")

In [ ]:
for index, row in gtzan_tracks.iterrows():
    for i in range(10):
        feature_dict = extract_ml_features(row, isGTZAN=True, chunk_number=i+1)
        if feature_dict is None:
            print("Broken audio file, continuing")
            continue
        flat_feature_dict = flatten_features(feature_dict)
        feature_row = pd.DataFrame(flat_feature_dict)
        gtzan_3sec_features_df = pd.concat([gtzan_3sec_features_df, feature_row], ignore_index=True)

for index, row in fma_tracks.iterrows():
    for i in range(10):
        feature_dict = extract_ml_features(row, isGTZAN=False, chunk_number=i+1)
        if feature_dict is None:
            print("Broken audio file, continuing")
            continue
        flat_feature_dict = flatten_features(feature_dict)
        feature_row = pd.DataFrame(flat_feature_dict)
        fma_3sec_features_df = pd.concat([fma_3sec_features_df, feature_row], ignore_index=True)

## Check DataFrame Shapes

In [ ]:
gtzan_3sec_features_df.shape, fma_3sec_features_df.shape

## Save the DataFrames

In [ ]:
fma_3sec_features_df.to_csv("data/features/fma/fma_3sec_features.csv")
gtzan_3sec_features_df.to_csv("data/features/gtzan/gtzan_3sec_features.csv")